In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

# Module 5: 모델 추론 (Inference)
## 배치 추론 & 실시간 추론 (Batch + Real-time Inference)

이 모듈에서는 Snowflake Model Registry에 등록된 **고객 LTV 예측 모델**을 활용하여  
다양한 방식으로 추론(Inference)을 수행하는 방법을 학습합니다.

### 학습 목표
- **Part 1**: 배치 추론 방식 이해 및 실습
  - SQL Native 배치 추론 (`mv.run`)
  - Dynamic Table 기반 자동 갱신 추론
  - 대용량 배치 추론 (`mv.run_batch` + Compute Pool)
- **Part 2**: 실시간 추론 서비스 구축
  - SPCS 기반 Inference Service 배포 (`mv.create_service`)
  - REST API 호출을 통한 실시간 예측
  - 서비스 관리 및 비용 최적화

### 추론 아키텍처 개요

```
[CUSTOMER_FEATURES 테이블]
         │
         ├─── Part 1: 배치 추론 ──────────────────────────────────────┐
         │    ① mv.run()         → CUSTOMER_LTV_SCORES 테이블 저장     │
         │    ② Dynamic Table    → LIVE_PREDICTED_LTVS 자동 갱신          │
         │    ③ mv.run_batch()   → SPCS Compute Pool 분산 처리        │
         │                                                              │
         └─── Part 2: 실시간 추론 ──────────────────────────────────  ┘
              ④ mv.create_service()      → SPCS Inference Service 배포
              ⑤ mv.run(..., service_name=...) → ms 단위 응답
```

**환경 정보:**
| 항목 | 값 |
|------|-----|
| Database | `DEMO` |
| Schema | `ML_DEMO` |
| Warehouse | `COMPUTE_WH` |
| 모델 | `CUSTOMER_LTV_PREDICTOR` V1 |
| Compute Pool | `SYSTEM_COMPUTE_POOL_CPU` (CPU_X64_S, 최대 150 노드) |

---
## Part 1: 배치 추론 (Batch Inference)

배치 추론은 **대량의 데이터**를 한 번에 처리하여 예측값을 생성하는 방식입니다.  
Snowflake는 SQL 네이티브 방식으로 모델 추론을 지원하므로, 별도의 인프라 없이  
Snowflake Warehouse 내에서 직접 모델을 실행할 수 있습니다.

| 방식 | 설명 | 권장 규모 |
|------|------|----------|
| `mv.run()` | SQL 네이티브 배치 추론 | ~ 수백만 건 |
| Dynamic Table | 자동 갱신 파이프라인 | 지속적 스트리밍 |
| `mv.run_batch()` | SPCS 분산 처리 | 수백만 건 이상 |

### 1. 환경 설정 (Setup)

Snowflake Session을 초기화하고, Model Registry에서 등록된 모델을 불러옵니다.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import snowflake.snowpark.functions as F
import pandas as pd
import time

# ─────────────────────────────────────────
# Snowflake Session 초기화
# ─────────────────────────────────────────
session = get_active_session()
session.use_database("DEMO")
session.use_schema("ML_DEMO")
session.use_warehouse("COMPUTE_WH")

print("✅ Session 초기화 완료")
print(f"   Database : {session.get_current_database()}")
print(f"   Schema   : {session.get_current_schema()}")
print(f"   Warehouse: {session.get_current_warehouse()}")

# ─────────────────────────────────────────
# Model Registry 연결 & 모델 버전 로드
# ─────────────────────────────────────────
# .version('V1')으로 버전명 지정 로드
# .alias('champion')으로 alias 지정 로드도 가능 (Module 5 참고)
reg = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")
mv  = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V1")

print("\n✅ 모델 로드 완료")
print(f"   모델명   : CUSTOMER_LTV_PREDICTOR")
print(f"   버전     : V1")

# 등록된 추론 함수 목록 확인 (predict / predict 모두 지원)
fns = mv.show_functions()
print(f"   등록 함수: {[f.get('name', f) if isinstance(f, dict) else str(f) for f in fns]}")

# ─────────────────────────────────────────
# 입력 테이블 구조 이해
# ─────────────────────────────────────────
# ✅ 추론 입력: CUSTOMER_FEATURES (원본 집계값 — 전처리 전)
#    모델은 OrdinalEncoder + StandardScaler + XGBRegressor 전체 Pipeline으로 등록됨
#    전처리가 Pipeline 내부에 포함되어 있으므로 원본 피처를 그대로 입력합니다.
#
# ℹ️  CUSTOMER_FEATURES_PROCESSED (스케일링/인코딩 완료본) 는
#    Module 2에서 탐색적 분석/검증 목적으로 생성된 테이블이며 추론에는 사용하지 않습니다.
print("\n📋 추론 입력 테이블: CUSTOMER_FEATURES (원본 집계, Pipeline이 내부 전처리 수행)")

### 2. SQL Native 배치 추론 (`mv.run`)

`mv.run()`은 Snowflake의 **SQL 네이티브** 방식으로 배치 추론을 수행합니다.  
데이터 이동 없이 Snowflake Warehouse 내에서 직접 모델이 실행되므로  
**높은 처리량과 낮은 지연 시간**을 보장합니다.

**특징:**
- Snowflake Warehouse에서 실행 (별도 인프라 불필요)
- Snowpark DataFrame 입력 지원
- 결과를 DataFrame으로 반환 → 바로 테이블에 저장 가능
- 동기(Synchronous) 실행 → 완료 후 즉시 결과 반환

In [ ]:
# ─────────────────────────────────────────
# CUSTOMER_FEATURES 로드 (원본 집계 — Pipeline이 내부적으로 전처리 수행)
# ─────────────────────────────────────────
print("📂 추론 입력 데이터 로드 (CUSTOMER_FEATURES)...")
customer_df    = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
customer_count = customer_df.count()

print(f"   총 고객 수    : {customer_count:,}명")
print(f"   특성 컬럼 수  : {len(customer_df.columns)}개")
print(f"   입력 컬럼 목록: {customer_df.columns}")

# ─────────────────────────────────────────
# predict — 예측 LTV 금액 (연속형)
# ─────────────────────────────────────────
print("\n🔮 배치 추론 실행 (function_name='predict')...")
start_time    = time.time()
predictions   = mv.run(customer_df, function_name="predict")
batch_time    = time.time() - start_time
print(f"   ✅ 완료 | 소요 시간: {batch_time:.2f}초")
print(f"   출력 컬럼: {predictions.columns}")
print(f"   처리 속도: {customer_count / batch_time:,.0f} records/sec")

In [ ]:
# ─────────────────────────────────────────
# predict 결과 → CUSTOMER_LTV_SCORES 저장
# ─────────────────────────────────────────
print("💾 예측 결과 저장 중...")

predictions.write.mode("overwrite").save_as_table(
    "DEMO.ML_DEMO.CUSTOMER_LTV_SCORES"
)

pred_cnt = session.table("DEMO.ML_DEMO.CUSTOMER_LTV_SCORES").count()
print(f"   ✅ CUSTOMER_LTV_SCORES 저장 완료 — {pred_cnt:,}건")

In [ ]:
# ─────────────────────────────────────────
# 예측 결과 샘플 확인
# ─────────────────────────────────────────
print("예측 결과 샘플 — 예측 LTV 상위 10건")
print("=" * 70)

# predictions 변수에서 직접 확인 (이전 셀에서 생성됨)
predictions.select(
    "C_CUSTKEY",
    "C_MKTSEGMENT",
    "TOTAL_ORDERS",
    "AVG_ORDER_VALUE",
    "FUTURE_12M_REVENUE",
    "PREDICTED_LABEL",
).sort(F.col("PREDICTED_LABEL").desc()).show(10)

print("\n컬럼 설명:")
print("   FUTURE_12M_REVENUE : 실제 LTV (정답)")
print("   PREDICTED_LABEL    : 모델 예측 LTV")


### 3. Dynamic Table 기반 자동 갱신 추론

**Dynamic Table**은 Snowflake의 선언적 데이터 파이프라인 기능으로,  
소스 테이블(`CUSTOMER_FEATURES`)에 새 데이터가 추가될 때 **자동으로 추론이 실행**됩니다.

#### 장점
- 별도의 스케줄러(Airflow, cron 등) **불필요**
- `TARGET_LAG`으로 갱신 주기 제어 (예: `'1 minute'`)
- 신규 고객 등록 → 자동 채점 → 결과 즉시 활용 가능
- Snowflake 내부의 완전한 **SQL 기반 ML 파이프라인** 구성

#### 작동 방식
```
신규 고객 등록
    │
    ▼
CUSTOMER_FEATURES 갱신
    │
    ▼ (최대 1분 내 자동 실행)
Dynamic Table 증분 새로고침
    │
    ▼
LIVE_PREDICTED_LTVS 자동 업데이트 완료
```

> **참고:** SQL에서 Registry 모델을 호출하려면  
> `MODEL(name)!PREDICT(col1, col2, ...)` 구문을 사용합니다.  
> Pipeline이 전처리를 내부적으로 수행하므로 **원본 CUSTOMER_FEATURES 컬럼을 그대로 전달**합니다.

In [ ]:
# ─────────────────────────────────────────
# Dynamic Table 생성: 자동 추론 파이프라인
#
# MODEL!PREDICT()는 스칼라 함수 → SELECT 절에서 직접 호출
# (TABLE() 래핑 사용 X — 테이블 함수가 아님)
# 반환값: OBJECT 타입 → :PREDICTED_LABEL 로 필드 접근
# ─────────────────────────────────────────
create_dt_sql = """
CREATE OR REPLACE DYNAMIC TABLE DEMO.ML_DEMO.LIVE_PREDICTED_LTVS
    TARGET_LAG = '1 minute'
    WAREHOUSE  = COMPUTE_WH
AS
SELECT
    cf.C_CUSTKEY,
    cf.C_MKTSEGMENT,
    cf.TOTAL_ORDERS,
    cf.AVG_ORDER_VALUE,
    cf.TOTAL_REVENUE,
    MODEL(DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR)!PREDICT(
        cf.C_MKTSEGMENT,
        cf.C_ACCTBAL,
        cf.TOTAL_ORDERS,
        cf.AVG_ORDER_VALUE,
        cf.TOTAL_REVENUE,
        cf.AVG_DISCOUNT,
        cf.AVG_QUANTITY,
        cf.DAYS_SINCE_LAST_ORDER,
        cf.C_NATIONKEY
    ):PREDICTED_LABEL::FLOAT AS PREDICTED_LTV,
    CURRENT_TIMESTAMP() AS SCORED_AT
FROM DEMO.ML_DEMO.CUSTOMER_FEATURES cf
"""
print("Dynamic Table 생성 중...")
session.sql(create_dt_sql).collect()
print("DEMO.ML_DEMO.LIVE_PREDICTED_LTVS 생성 완료")
# Dynamic Table 상태 확인
print("\nDynamic Table 정보:")
session.sql("SHOW DYNAMIC TABLES LIKE 'LIVE_PREDICTED_LTVS' IN SCHEMA DEMO.ML_DEMO").show()
# 결과 확인
print("\n예측 결과 샘플:")
session.table("DEMO.ML_DEMO.LIVE_PREDICTED_LTVS").show(5)

> **동작 원리 요약:**
>
> 1. Snowflake가 `CUSTOMER_FEATURES` 테이블 변경을 자동 감지
> 2. `TARGET_LAG = '1 minute'` 기준으로 최대 1분 내 증분 새로고침 실행
> 3. `CUSTOMER_LTV_PREDICTOR!PREDICT()` SQL 함수로 신규 고객 자동 채점
> 4. `LIVE_PREDICTED_LTVS` 테이블이 항상 최신 예측값 유지
>
> 이 방식은 **배치 ETL 파이프라인과 ML 추론을 통합**하는 가장 간단한 방법입니다.  
> 별도의 Airflow DAG, Lambda, 혹은 스케줄 작업이 전혀 필요하지 않습니다.

### 4. 대용량 배치 추론 (`mv.run_batch`)

수백만 건 이상의 대용량 데이터를 처리할 때는 `mv.run_batch()`를 사용합니다.  
**Snowpark Container Services (SPCS) Compute Pool**을 활용하여  
여러 노드에서 병렬 분산 처리로 대용량 추론 작업을 수행합니다.

#### mv.run() vs mv.run_batch() 비교

| 항목 | `mv.run()` | `mv.run_batch()` |
|------|-----------|-----------------|
| 실행 방식 | 동기 (Synchronous) | 비동기 (Asynchronous) |
| 실행 환경 | Snowflake Warehouse | SPCS Compute Pool |
| 권장 규모 | ~ 수백만 건 | 수백만 건 이상 |
| 반환값 | DataFrame | Job 객체 → `.result()` |
| 확장성 | Warehouse 크기 의존 | 최대 150 노드 자동 스케일링 |

#### Compute Pool 정보
- Pool 이름: `SYSTEM_COMPUTE_POOL_CPU`
- 인스턴스 타입: `CPU_X64_S`
- 최대 노드 수: 150개

In [ ]:
# ─────────────────────────────────────────
# 대용량 배치 추론 — 비동기 Job 실행 (SPCS)
#
# run_batch는 SPCS Compute Pool에서 비동기로 실행됩니다.
# output_spec: 결과를 저장할 Stage 경로 (필수)
# ─────────────────────────────────────────
from snowflake.ml.model.batch import OutputSpec

# Stage 준비
session.sql("CREATE STAGE IF NOT EXISTS DEMO.ML_DEMO.ML_BATCH_OUTPUT").collect()
session.sql("REMOVE @DEMO.ML_DEMO.ML_BATCH_OUTPUT/ltv_predictions/").collect()

print("대용량 배치 추론 시작 (SPCS Compute Pool)...")
print(f"   Compute Pool : SYSTEM_COMPUTE_POOL_CPU")
print(f"   데이터 규모  : {customer_count:,}건")

job = mv.run_batch(
    X=customer_df,
    compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    output_spec=OutputSpec(
        stage_location="@DEMO.ML_DEMO.ML_BATCH_OUTPUT/ltv_predictions/"
    )
)

print(f"   Job 제출 완료 (비동기 실행 중)")

# Job 완료 대기
print("\nJob 완료까지 대기 중...")
import time
batch_start = time.time()
job.wait()
elapsed = time.time() - batch_start

print(f"\n대용량 배치 추론 완료!")
print(f"   총 소요 시간 : {elapsed:.1f}초")
print(f"   처리 속도    : {customer_count / elapsed:,.0f} records/sec")

# 결과 확인 (Parquet 파일로 저장됨 — _SUCCESS 마커 파일 제외)
print("\n결과 확인:")
session.sql("REMOVE @DEMO.ML_DEMO.ML_BATCH_OUTPUT/ltv_predictions/_SUCCESS").collect()
result_df = session.read.parquet("@DEMO.ML_DEMO.ML_BATCH_OUTPUT/ltv_predictions/")
print(f"   결과 건수: {result_df.count():,}건")
result_df.show(5)


---
## Part 2: 실시간 추론 (Real-time Inference via SPCS)

**Snowpark Container Services (SPCS)**를 활용하여 항상 실행 중인  
**Inference Service**를 배포하면, **밀리초(ms) 단위**의 실시간 예측이 가능합니다.

#### 배치 추론 vs 실시간 추론

| 구분 | 배치 추론 | 실시간 추론 |
|------|----------|------------|
| 응답 시간 | 초 ~ 분 단위 | 밀리초(ms) 단위 |
| 처리 방식 | 대량 일괄 처리 | 단건/소규모 즉시 처리 |
| 인프라 | Warehouse (공유) | SPCS Service (전용) |
| 비용 | 실행 시간만 과금 | 서비스 운영 기간 과금 |
| 적합한 사례 | 야간 배치, 주기적 재채점 | 앱 API, CRM, 실시간 개인화 |

#### 실시간 추론 활용 사례
- 고객 앱에서 **즉시 개인화 추천** 생성
- 웹사이트 방문 시 **실시간 LTV 예측**
- 영업팀 CRM 시스템의 **실시간 고객 등급 표시**
- 결제 시 즉시 **프리미엄 서비스 업셀링** 여부 결정

### 5. Inference Service 배포

`mv.create_service()`를 사용하여 모델을 SPCS에 배포합니다.  
배포된 서비스는 **REST API 엔드포인트**를 제공하며,  
Snowflake 내부 및 외부에서 HTTP 요청으로 호출 가능합니다.

#### 배포 옵션
| 파라미터 | 설명 |
|---------|------|
| `service_name` | 서비스 이름 (스키마 내 고유) |
| `service_compute_pool` | 서비스 실행 Compute Pool |
| `ingress_enabled` | 외부 HTTP 엔드포인트 활성화 여부 |
| `gpu_requests` | GPU 수 (CPU 모델은 None) |

> ⏱️ **배포 소요 시간:** 컨테이너 이미지 빌드 및 초기화로 인해 **약 2~5분** 소요됩니다.

In [ ]:
# ─────────────────────────────────────────
# SPCS Inference Service 배포 (create_service)
# ─────────────────────────────────────────

# 기존 서비스가 있으면 삭제 후 재배포
try:
    mv.delete_service("LTV_INFERENCE_SVC")
    print("기존 LTV_INFERENCE_SVC 서비스 삭제 완료")
except Exception:
    pass  # 서비스가 없는 경우 무시

print("🚀 Inference Service 배포 중...")
print("   서비스명    : LTV_INFERENCE_SVC")
print("   Compute Pool: SYSTEM_COMPUTE_POOL_CPU")
print("   예상 소요   : 2~5분 (컨테이너 이미지 빌드 및 초기화)")
print()

mv.create_service(
    service_name="LTV_INFERENCE_SVC",
    service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    gpu_requests=None,
)

print("✅ Inference Service 배포 완료!")
print()
print("💡 SPCS 서비스 특징:")
print("   - 자동 스케일링  : 요청량 증가 시 워커 수 자동 확장")
print("   - 고가용성       : 컨테이너 장애 시 자동 재시작")
print("   - 네트워크 격리  : Snowflake VNet 내부 통신")
print("   - 멀티 버전 지원 : 동일 모델의 여러 버전 동시 서비스 가능")


### 6. Service 상태 확인

배포된 서비스의 상태를 확인합니다.  
서비스가 **`RUNNING`** 상태여야 추론 요청을 처리할 수 있습니다.

| 상태 | 설명 |
|------|------|
| `PENDING` | 컨테이너 초기화 중 |
| `RUNNING` | 정상 운영 중 (추론 요청 처리 가능) |
| `SUSPENDED` | 일시 중지 상태 |
| `FAILED` | 오류 발생 (로그 확인 필요) |

In [ ]:
# Python API로 현재 모델의 서비스 목록 확인
print("📋 CUSTOMER_LTV_PREDICTOR V1의 배포된 서비스 목록:")
print("=" * 60)

services = mv.list_services()
print(services)

In [ ]:
# SQL로 스키마 내 모든 서비스 상태 확인
print("📋 DEMO.ML_DEMO 스키마의 모든 서비스:")
print("=" * 70)
session.sql("SHOW SERVICES IN SCHEMA DEMO.ML_DEMO").show()

# 서비스 상태가 RUNNING인지 확인
print("\n🔍 LTV_INFERENCE_SVC 상태:")
status_df = session.sql("""
    SELECT
        "name"            AS SERVICE_NAME,
        "status"          AS STATUS,
        "compute_pool"    AS COMPUTE_POOL,
        "min_instances"   AS MIN_INSTANCES,
        "max_instances"   AS MAX_INSTANCES,
        "created_on"      AS CREATED_ON
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
    WHERE "name" = 'LTV_INFERENCE_SVC'
""")
status_df.show()

### 7. REST API 호출 (실시간 추론)

서비스가 `RUNNING` 상태가 되면, `mv.run()`에 `service_name` 파라미터를 추가하여  
**실시간 추론**을 수행할 수 있습니다.

Snowpark Python SDK가 내부적으로 SPCS REST API를 호출하므로  
별도의 HTTP 클라이언트 코드 없이 동일한 `mv.run()` 인터페이스를 사용합니다.

```python
# 배치 추론 (Warehouse)
mv.run(df, function_name="predict")

# 실시간 추론 (SPCS Service)
mv.run(df, function_name="predict", service_name="LTV_INFERENCE_SVC")
```

In [ ]:
# ─────────────────────────────────────────
# 실시간 추론용 테스트 샘플 준비 (단건 ~ 소규모 배치)
# ─────────────────────────────────────────
print("🔬 실시간 추론 테스트 준비 중...")
test_sample = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES").limit(10)
test_count  = test_sample.count()
print(f"   테스트 샘플 수: {test_count}건\n")

# ─────────────────────────────────────────
# 실시간 추론 실행 (SPCS Service)
# ─────────────────────────────────────────
print("⚡ 실시간 추론 실행 (SPCS Service)...")
rt_start = time.time()

realtime_preds = mv.run(
    test_sample,
    function_name="predict",
    service_name="LTV_INFERENCE_SVC"
)

rt_ms = (time.time() - rt_start) * 1000
print(f"   ✅ 실시간 추론 완료")
print(f"   응답 시간: {rt_ms:.1f} ms")
print()
realtime_preds.show()

### 8. 외부 REST API 호출 (External HTTP Endpoint)

`create_service(ingress_enabled=True)`로 배포하면 **외부 HTTP 엔드포인트**가 생성됩니다.  
이를 통해 Snowflake 외부의 앱(웹/모바일), CRM, 마케팅 자동화 툴 등에서 직접 예측을 요청할 수 있습니다.

#### 인증 방식
- **Programmatic Access Token (PAT)**: 가장 간단한 인증 방법
- Header에 `Authorization: Snowflake Token="<PAT_TOKEN>"` 전달

#### 요청 형식
```
POST https://<service-id>-<account-id>.snowflakecomputing.app/predict
Header: Authorization: Snowflake Token="<YOUR_PAT>"
Body:   {"data": [["BUILDING", 711.56, 12, 186000.5, 2232006.0, 0.04, 25.2, 120, 15]]}
```

각 배열 원소는 모델 입력 컬럼 순서대로:  
`C_MKTSEGMENT, C_ACCTBAL, TOTAL_ORDERS, AVG_ORDER_VALUE, TOTAL_REVENUE, AVG_DISCOUNT, AVG_QUANTITY, DAYS_SINCE_LAST_ORDER, C_NATIONKEY`

In [ ]:
# ─────────────────────────────────────────
# 외부 REST API 호출 샘플 코드
# (실제 실행하려면 PAT 토큰과 엔드포인트 URL 필요)
# ─────────────────────────────────────────

# 1. 엔드포인트 URL 확인
print("📡 Inference Service 엔드포인트 확인")
print("=" * 60)
endpoints_df = session.sql("SHOW ENDPOINTS IN SERVICE DEMO.ML_DEMO.LTV_INFERENCE_SVC")
endpoints_df.show()

# 2. 외부 호출 예시 (Python requests)
print("\n📋 외부 REST API 호출 예시 코드:")
print("-" * 60)
sample_code = '''
import requests

# 엔드포인트 URL (위 SHOW ENDPOINTS 결과의 ingress_url)
url = "https://<service-id>-<account-id>.snowflakecomputing.app/predict"

# PAT 토큰 (Snowsight > User Menu > Programmatic Access Tokens)
headers = {
    "Authorization": 'Snowflake Token="<YOUR_PAT_TOKEN>"',
    "Content-Type": "application/json"
}

# 입력 데이터: [C_MKTSEGMENT, C_ACCTBAL, TOTAL_ORDERS, AVG_ORDER_VALUE,
#              TOTAL_REVENUE, AVG_DISCOUNT, AVG_QUANTITY, DAYS_SINCE_LAST_ORDER, C_NATIONKEY]
payload = {
    "data": [
        ["BUILDING", 711.56, 12, 186000.5, 2232006.0, 0.04, 25.2, 120, 15],
        ["AUTOMOBILE", 3456.78, 25, 210000.0, 5250000.0, 0.05, 30.1, 45, 7]
    ]
}

response = requests.post(url, json=payload, headers=headers)
print(response.json())
# 응답: {"data": [[predicted_ltv_1], [predicted_ltv_2]]}
'''
print(sample_code)

In [ ]:
# ─────────────────────────────────────────
# 배치 vs 실시간 응답 시간 비교
# ─────────────────────────────────────────
print("📊 추론 방식별 응답 시간 비교 (테스트 샘플: 10건)")
print("=" * 55)

# ① 배치 추론 (Warehouse)
print("\n① 배치 추론 (mv.run — Snowflake Warehouse)")
b_start = time.time()
_       = mv.run(test_sample, function_name="predict")
b_ms    = (time.time() - b_start) * 1000
print(f"   응답 시간: {b_ms:8.1f} ms")

# ② 실시간 추론 (SPCS Service)
print("\n② 실시간 추론 (mv.run — SPCS Service)")
r_start = time.time()
_       = mv.run(test_sample, function_name="predict", service_name="LTV_INFERENCE_SVC")
r_ms    = (time.time() - r_start) * 1000
print(f"   응답 시간: {r_ms:8.1f} ms")

# 비교 결과
print()
print("─" * 55)
print(f"   배치 추론   : {b_ms:8.1f} ms")
print(f"   실시간 추론 : {r_ms:8.1f} ms")
ratio = b_ms / r_ms if r_ms > 0 else float('inf')
print(f"   속도 비율   : 실시간이 {ratio:.1f}x 빠름 (소규모 기준)")
print()
print("💡 선택 기준:")
print("   소규모 단건~수십 건 → 실시간 서비스 (ms 응답)")
print("   대용량 일괄 처리    → 배치 추론 (Warehouse 또는 run_batch)")

### 9. Service 삭제 (비용 관리)

SPCS Inference Service는 실행 중인 동안 **Compute Pool 크레딧**을 소비합니다.  
사용하지 않는 서비스는 삭제하여 불필요한 비용을 방지합니다.

#### 비용 관리 전략

| 상황 | 권장 조치 |
|------|---------|
| 개발/테스트 완료 후 | 서비스 즉시 삭제 (`mv.delete_service()`) |
| 야간 트래픽 없음 | Compute Pool Suspend (`ALTER COMPUTE POOL ... SUSPEND`) |
| 운영 환경 상시 운영 | `min_instances=0` 설정으로 유휴 시 자동 축소 |
| 비용 모니터링 | Snowflake Credit Usage Dashboard 활용 |

> ⚠️ **주의:** 서비스 삭제 후 실시간 추론은 불가합니다.  
> 재배포 시 `mv.create_service()`를 다시 실행해야 합니다.

In [ ]:
# ─────────────────────────────────────────
# 삭제 전 서비스 상태 최종 확인
# ─────────────────────────────────────────
print("📋 삭제 전 서비스 목록:")
print(mv.list_services())

# ─────────────────────────────────────────
# SPCS Inference Service 삭제
# ─────────────────────────────────────────
print("\n🗑️  Inference Service 삭제 중...")
print("   서비스명: LTV_INFERENCE_SVC\n")

mv.delete_service("LTV_INFERENCE_SVC")

print("✅ LTV_INFERENCE_SVC 서비스 삭제 완료")

# ─────────────────────────────────────────
# Compute Pool 상태 확인 및 관리
# ─────────────────────────────────────────
print("\n💡 Compute Pool 관리 명령어:")
print("   # Compute Pool 상태 확인")
print("   SHOW COMPUTE POOLS LIKE 'SYSTEM_COMPUTE_POOL_CPU';")
print()
print("   # Compute Pool 일시 중지 (비용 절감)")
print("   ALTER COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU SUSPEND;")
print()
print("   # Compute Pool 재개")
print("   ALTER COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU RESUME;")
print()

# Compute Pool 현재 상태 확인
session.sql("SHOW COMPUTE POOLS LIKE 'SYSTEM_COMPUTE_POOL_CPU'").show()

---
## Module 5 요약

### 추론 방식 선택 가이드

| 요구사항 | 권장 방식 | 명령 |
|---------|---------|------|
| 전체 고객 정기 재채점 | SQL Native 배치 | `mv.run(df)` → 테이블 저장 |
| 신규 고객 자동 채점 | Dynamic Table | `LIVE_PREDICTED_LTVS` 자동 갱신 |
| 수천만 건 이상 처리 | 분산 배치 | `mv.run_batch(df, compute_pool=...)` |
| 앱/API 실시간 응답 | SPCS Service | `mv.create_service()` + `service_name=` |

### 주요 명령어 정리

```python
# ── 배치 추론 ───────────────────────────────────────────────────────────
predictions = mv.run(customer_df, function_name="predict")

# ── 대용량 배치 (비동기) ─────────────────────────────────────────────────
job = mv.run_batch(customer_df, function_name="predict",
                   compute_pool="SYSTEM_COMPUTE_POOL_CPU")
job.wait()
results = job.result()

# ── SPCS 실시간 서비스 배포 ──────────────────────────────────────────────
mv.create_service(service_name="LTV_INFERENCE_SVC",
                  service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
                  ingress_enabled=True, gpu_requests=None)

# ── 실시간 추론 ──────────────────────────────────────────────────────────
rt_preds = mv.run(test_df, function_name="predict",
                  service_name="LTV_INFERENCE_SVC")

# ── 서비스 삭제 (비용 관리) ──────────────────────────────────────────────
mv.delete_service("LTV_INFERENCE_SVC")
```

### 다음 단계

- **Module 6**: ML Jobs & Pipeline Orchestration (스케줄링, Task Graph)
- **Module 7**: 모델 모니터링 (Drift Detection, 성능 지표 추적)
- **모델 재학습**: 새 데이터 누적 시 Module 4~5 파이프라인 재실행
- **A/B 테스트**: Registry에 V2 등록 후 트래픽 분산 운영

---
*Snowflake Notebooks (Container Runtime) | DEMO.ML_DEMO | CUSTOMER_LTV_PREDICTOR V1*